# 04 — Consensus & Evaluate

This notebook:

1. Runs consensus voting on matches from notebook 03
2. Detects speed zone boundaries within segments
3. Compares our estimates against Overture's own normalized `speed_limits` values
4. Computes accuracy metrics and generates an evaluation report
5. Shows a final comparison map

In [ ]:
import os
import geopandas as gpd
from slc import consensus, evaluate, fetch, viz

BBOX = (-111.920, 40.855, -111.855, 40.910)

In [ ]:
matches = gpd.read_parquet('matches.parquet')
overture_raw = fetch.fetch_overture_segments(BBOX)
overture = fetch.extract_overture_speed_limits(overture_raw)

In [ ]:
# Compute consensus
estimates = consensus.compute_consensus(matches)
print(f'Segments with estimates: {len(estimates)}')
print(f'Conflicted segments: {estimates["has_conflict"].sum()}')
print(estimates['speed_mph'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
estimates['confidence'].hist(bins=20)
plt.xlabel('Confidence score')
plt.title('Distribution of estimate confidence scores')
plt.show()

In [ ]:
# Compare against Overture's built-in speed limits (already normalized)
comparison = evaluate.compare_to_overture(estimates, overture)
metrics = evaluate.compute_metrics(comparison)
print(metrics)

In [ ]:
report = evaluate.generate_report(comparison, metrics)
print(report)

In [ ]:
# Final comparison map
viz.map_comparison(comparison, overture)

In [ ]:
# Export final estimates
estimates.to_parquet('estimates.parquet', index=False)
print('Saved estimates.parquet')